# DistilBERT emotion model: GPU training run (Colab)

Training happens here on the GPU; only the resulting **int8 ONNX model** goes to the edge device for inference.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Expect about 15-25 min: a 4-run validation sweep (lr × epochs), the final gated run (≥ 90 % rounded test accuracy), then vocabulary pruning and int8 export.
3. The last cell downloads `emotion_text_results.zip`. Send it back, or commit `results/` and `docs/` to the repo.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone -b claude/fervent-sagan-71ls6g https://github.com/unexperiencedone/lightweight-emotion-detection.git
%cd lightweight-emotion-detection
!pip install -q -r requirements.txt

In [ ]:
import os; os.environ['PYTHONPATH'] = os.getcwd()
!bash scripts/run_text.sh 2>&1 | grep -v -i warning

In [ ]:
import json
r = json.load(open('artifacts/text/results.json'))
print('gate:', r['gate'])
for k, v in r['variants'].items():
    print(f"{k:24s} acc {v['accuracy_rounded_pct']:5.1f}%  macroF1 {v['macro_f1']:.4f}  size {v.get('size_mb', float('nan')):7.1f} MB")
print('deploy choice:', r.get('deploy_choice'))
print('note: edge latency measured in this notebook is Colab CPU latency, not your device')

In [ ]:
# package what the edge device and the docs need (int8 model + tokenizer + remap + results), not the training checkpoints
!zip -q -r emotion_text_results.zip artifacts/text/results.json artifacts/text/logits.npz artifacts/text/vocab_remap.npy \
    artifacts/text/model_pruned_int8_emb8.onnx artifacts/text/model_int8.onnx artifacts/text/torch_fp32/tokenizer* \
    artifacts/text/torch_fp32/vocab.txt artifacts/text/torch_fp32/special_tokens_map.json results docs
from google.colab import files
files.download('emotion_text_results.zip')